<h1 align=center style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
استخرمون نشه؟</font>
</h1>

<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
مقدمه و صورت مسئله
</font>
</h2>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size=3>
در بسیاری از وب‌سایت‌های گردشگری و رزرو اقامتگاه‌ها، کاربران می‌توانند تصاویر مختلفی از اقامتگاه‌های مختلف را مشاهده کنند. در این مسئله قصد داریم با استفاده از مجموعه‌ای از تصاویر اقامتگاه‌ها، مدلی طراحی کنیم که بتواند بر اساس تحلیل تصاویر یک اقامتگاه، وجود یا عدم وجود استخر در آن اقامتگاه را پیش‌بینی کند.
</font>
</p>

<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
وارد کردن کتابخانه‌های مورد نیاز
</font>
</h2>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size=3>
    ابتدا کتابخانه‌های مورد نیازتان را وارد کنید.
</font>
</p>

In [7]:
import pandas as pd
import numpy as np
import os
import glob

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

I0000 00:00:1789811513.666754 3045454 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1789811513.980096 3045454 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1789811515.476652 3045454 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
معرفی مجموعه داده
</font>
</h2>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size=3>
مجموعه داده ارائه‌شده به صورت JSON حاوی اطلاعاتی درباره ۱۵۹۹ اقامتگاه است که شامل ویژگی‌های مختلفی می‌باشند. این مجموعه شامل یک لیست با کلید "rooms" است که اقامتگاه‌های موجود در آن قرار دارند. هر رکورد در این مجموعه داده نمایانگر یک اقامتگاه با خصوصیات و امکانات مربوط به آن است. در جدول زیر، کلیدها و توضیحات آن‌ها آورده شده‌اند.
</font>
</p>

<center>
<div dir=rtl style="direction: rtl;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazir" size=3>

|کلید|توضیحات|
|:------:|:---:|
|id|شناسه یکتای اقامتگاه|
|title|عنوان اقامتگاه|
|description|توضیحاتی در مورد اقامتگاه|
|province|اطلاعات استان شامل شناسه و نام استان|
|city|اطلاعات شهر شامل شناسه و نام شهر|

</font>
</div>
</center>


<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size=3>
نمونه‌ای از این داده‌ها به صورت زیر است:
</font>
</p>

```json
{
  "rooms": [
    {
      "id": 3202100,
      "title": "اجاره سوئیت در جاده جواهرده - اشکونکوه",
      "description": "این سوئیت بدون اتاق خواب با بالکنی دلباز و چشم اندازی زیبا در طبقه همکف یک ساختمان دو طبقه واقع شده است. فاصله اقامتگاه تا سوپرمارکت و نانوایی به ترتیب حدود ۵۰ و ۵۰۰ متر می باشد.",
      "province": {
        "id": "p26",
        "name": "مازندران"
      },
      "city": {
        "id": 303,
        "name": "رامسر"
      },
    },
    ...
  ]
}
```


<h2 align="right" style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
خواندن مجموعه داده
</font>
</h2>

<p dir="rtl" style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size="3">
    ابتدا نیاز است فایل‌های مجموعه‌‌ی داده را بخوانید. ویژگی‌های آموزشی در فایل <code>train.json</code> و دایرکتوری مربوط به تصاویر نمونه‌های آزمون که باید دسته‌ی آن‌ها را پیش‌بینی کنید، در فایل <code>test.json</code> ذخیره شده‌اند. در صورت نیاز می‌توانید بخشی از داده‌های آموزشی را به عنوان داده‌های اعتبارسنجی جدا کنید. توجه داشته باشید که ستون‌های <code>description</code> و <code>title</code> از داده‌های آزمون حذف شده‌اند و هدف اصلی این فایل، دسترسی به دایرکتوری تصاویر داده‌های آزمون است. دایرکتوری هر تصویر برابر با <code>id</code> هر اقامتگاه است و داخل پوشه <code>pictures</code> قرار دارد.
</font>
</p>

<p dir="rtl" style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
    <font face="vazir">
        <span style="color:orange"><b>نکته:</b></span>
        حجم فایل‌های این سوال حدود ۳.۸ گیگابایت است. در صورتی‌که بارگذاری این فایل‌ها در محیط کولب برای‌تان دشوار است می‌توانید با استفاده از دستور زیر این دو فایل را به‌صورت مستقیم در محیط کولب بارگیری کنید.
        همچنین می‌توانید از لینک درایو زیر فایل‌ها را دانلود کرده و بر روی محیط دستگاه خود از آن استفاده کنید.
    </font>
</p>

<p dir="rtl" style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
    <font face="vazir">
        <a href="https://drive.google.com/file/d/1b8O_a6ywcsbLqJAGDGCkePrdn1cFlXl0/view?usp=sharing" target="_blank" style="color:blue;">لینک دانلود از گوگل درایو</a>
    </font>
</p>





In [ ]:
# # Run this cell to download directly to colab\

# # Install gdown
# !pip install gdown

# # Import necessary libraries
# import gdown
# import zipfile

# # Google Drive file ID and destination filename
# file_id = '1b8O_a6ywcsbLqJAGDGCkePrdn1cFlXl0'
# destination = 'downloaded_file.zip'

# # Download the file from Google Drive
# gdown.download(f'https://drive.google.com/uc?id={file_id}', destination, quiet=False)

# # Unpack the zip file
# with zipfile.ZipFile(destination, 'r') as zip_ref:
#     zip_ref.extractall('unzipped_content')

# print("Download and extraction complete.")


In [2]:
train_data = pd.read_json('data/train.json')
print(train_data)

                                                  rooms
0     {'id': 3175858, 'title': 'رزرو ویلا با استخر ر...
1     {'id': 3237321, 'title': 'اجاره ویلا استخردار ...
2     {'id': 3154228, 'title': 'اجاره ویلا با استخر ...
3     {'id': 3169850, 'title': 'رزرو سوئیت در جواهرد...
4     {'id': 3207557, 'title': 'اجاره سوئیت در جاده ...
...                                                 ...
1594  {'id': 3167459, 'title': 'اجاره ویلا استخردار ...
1595  {'id': 3207406, 'title': 'اجاره ویلا استخردار ...
1596  {'id': 3172620, 'title': 'اجاره ویلا چابکسر - ...
1597  {'id': 3237948, 'title': 'اجاره سوئیت در جواهر...
1598  {'id': 3229812, 'title': 'اجاره خانه با ویو کو...

[1599 rows x 1 columns]


In [3]:
test_data = pd.read_json('data/test.json')
print(test_data)

               rooms
0    {'id': 3160664}
1    {'id': 3195184}
2    {'id': 3224078}
3    {'id': 3233712}
4    {'id': 3201449}
..               ...
856  {'id': 3232618}
857  {'id': 3230930}
858  {'id': 3171083}
859  {'id': 3168538}
860  {'id': 3168123}

[861 rows x 1 columns]


<h2 align="right" style="line-height: 200%; font-family: Vazir; color: #0099cc;">
    پیش‌پردازش و مهندسی ویژگی
</h2>

<p dir="rtl" style="direction: rtl; text-align: justify; line-height: 200%; font-family: Vazir; font-size: medium;">
    در این سؤال، باید با استفاده از توصیفات و تیتر هر اقامت‌گاه، وجود یا عدم وجود استخر را استخراج کنید. می‌توانید از هر تکنیک پیش‌پردازش/مهندسی ویژگی دلخواهتان، استفاده کنید.
    <br>
    تکنیک‌هایی که استفاده می‌کنید به شکل مستقیم توسط سامانه داوری ارزیابی نمی‌شوند، بلکه همه آن‌ها بر دقت مدل شما تأثیر خواهند گذاشت. بنابراین، هرچه پیش‌پردازش و مهندسی ویژگی بهتری انجام دهید و دقت مدل بهبود یابد، امتیاز بیشتری از این سؤال کسب خواهید کرد.
</p>


In [ ]:
df_train = pd.json_normalize(train_data['rooms'])
df_test = pd.json_normalize(test_data['rooms'])

def extract_pool_label(row):
    text = str(row.get('title', '')) + " " + str(row.get('description', ''))
    
    pool_keywords = ['استخر', 'جکوزی', 'آبگرم', 'آب گرم']
    
    for kw in pool_keywords:
        if kw in text.lower():
            return True
    return False

df_train['Pool'] = df_train.apply(extract_pool_label, axis=1)

print(df_train['Pool'].value_counts())

Pool
True     933
False    666
Name: count, dtype: int64


In [9]:
TRAIN_DIR = 'data/train'
TEST_DIR = 'data/test'

def create_image_dataframe(df, base_dir, has_label=True):
    img_paths = []
    ids = []
    labels = []
    
    for index, row in df.iterrows():
        room_id = str(row['id'])
        room_dir = os.path.join(base_dir, room_id)
        
        if os.path.exists(room_dir):
            for img_name in os.listdir(room_dir):
                if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                    img_paths.append(os.path.join(room_dir, img_name))
                    ids.append(row['id'])
                    if has_label:
                        labels.append(row['Pool'])
                        
    if has_label:
        return pd.DataFrame({'id': ids, 'image_path': img_paths, 'Pool': labels})
    else:
        return pd.DataFrame({'id': ids, 'image_path': img_paths})

train_img_df = create_image_dataframe(df_train, base_dir=TRAIN_DIR, has_label=True)
test_img_df = create_image_dataframe(df_test, base_dir=TEST_DIR, has_label=False)

print(f"train_img_df : {len(train_img_df)}")
print(f"test_img_df : {len(test_img_df)}")

if len(train_img_df) > 0:
    train_img_df['Pool_str'] = train_img_df['Pool'].astype(str)

train_img_df : 15841
test_img_df : 8505


<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
مدل‌سازی
</font>
</h2>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size=3>
    حال که داده‌ها را پیش‌پردازش کرده‌اید، و دسته‌بندی وجود یا عدم وجود استخر را از متن انجام داده‌اید، وقت آن است که مدلی آموزش دهید که بتواند متغیر هدف این مسئله یعنی وجود یا عدم وجود استخر را پیش‌بینی کند.
    <br>
    شما مجاز هستید از هر مدلی که آموخته‌اید استفاده کنید. هدف این سوال پیش‌بینی هرچه بهتر متغیر هدف است! که آن همان وجود یا عدم وجود استخر است!
</font>
</p>

In [10]:
datagen = ImageDataGenerator(validation_split=0.2, rescale=1./255)

train_generator = datagen.flow_from_dataframe(
    dataframe=train_img_df,
    x_col='image_path',
    y_col='Pool_str',
    target_size=(128, 128),
    batch_size=32,
    class_mode='binary',
    subset='training'
)

val_generator = datagen.flow_from_dataframe(
    dataframe=train_img_df,
    x_col='image_path',
    y_col='Pool_str',
    target_size=(128, 128),
    batch_size=32,
    class_mode='binary',
    subset='validation',
    shuffle=False
)

base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(128, 128, 3))
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(64, activation='relu')(x)
predictions = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=predictions)
model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])

model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=3
)

Found 12673 validated image filenames belonging to 2 classes.
Found 3168 validated image filenames belonging to 2 classes.


I0000 00:00:1789811745.312046 3045454 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 4035 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
Epoch 1/3


I0000 00:00:1789811750.835925 3045454 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.
I0000 00:00:1789811753.327665 3053007 service.cc:153] XLA service 0x77fa68032700 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1789811753.327683 3053007 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 4050 Laptop GPU, Compute Capability 8.9 (Driver: 13.0.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.22.0)
I0000 00:00:1789811753.406525 3053007 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1789811754.010065 3053007 cuda_dnn.cc:461] Loaded cuDNN version 92200
I0000 00:00:1789811754.038060 3053007 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_8022__.114
E0000 00:00:1789811760.223597 3053007 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup e

  1/397 ━━━━━━━━━━━━━━━━━━━━ 1:21:50 12s/step - accuracy: 0.6250 - loss: 0.7643

I0000 00:00:1789811763.174179 3053007 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


115/397 ━━━━━━━━━━━━━━━━━━━━ 53s 189ms/step - accuracy: 0.6364 - loss: 0.6590

E0000 00:00:1789811786.454142 3053009 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


397/397 ━━━━━━━━━━━━━━━━━━━━ 110s 246ms/step - accuracy: 0.6809 - loss: 0.5865 - val_accuracy: 0.7004 - val_loss: 0.5644
Epoch 2/3
397/397 ━━━━━━━━━━━━━━━━━━━━ 84s 213ms/step - accuracy: 0.7183 - loss: 0.5366 - val_accuracy: 0.7247 - val_loss: 0.5432
Epoch 3/3
397/397 ━━━━━━━━━━━━━━━━━━━━ 87s 219ms/step - accuracy: 0.7362 - loss: 0.5081 - val_accuracy: 0.7175 - val_loss: 0.5534


<h3 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
معیار ارزیابی
</font>
</h3>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size=3>
    معیاری که برای ارزیابی عملکرد مدل انتخاب کرده‌ایم، <code>f1_score</code> نام دارد.
    <br>
    این معیار، سنجه ارزیابی کیفیت مدل شماست. به عبارت بهتر در سامانه داوری هم از همین معیار برای نمره‌دهی استفاده شده است.
    <br>
    پیشنهاد می‌شود با توجه به این معیار، عملکرد مدل خود را بر روی مجموعه داده آموزش یا اعتبارسنجی ارزیابی کنید.
    
</font>
</p>

In [11]:
from sklearn.metrics import  f1_score, classification_report, confusion_matrix
val_generator.reset()
preds = model.predict(val_generator)
y_pred = (preds > 0.5).astype(int).flatten()

y_true = val_generator.classes

print("F1 Score:", f1_score(y_true, y_pred))
print(classification_report(y_true, y_pred))


99/99 ━━━━━━━━━━━━━━━━━━━━ 19s 173ms/step
F1 Score: 0.7257125344774747
              precision    recall  f1-score   support

           0       0.63      0.80      0.71      1354
           1       0.82      0.65      0.73      1814

    accuracy                           0.72      3168
   macro avg       0.73      0.73      0.72      3168
weighted avg       0.74      0.72      0.72      3168



<h2 align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
 پیش‌بینی برای داده تست و خروجی
</font>
</h2>

<p dir=rtl style="direction: rtl;text-align: right;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazir" size=3>
    پس از مهندسی ویژگی و مدلسازی، الگوریتمی دارید که می‌تواند شما را از متغیرهای مستقل به متغیر هدف برساند.
    <br>
    از این مدل برای پیش‌بینی نمونه‌های موجود در داده تست استفاده کنید و نتایج را در قالب جدول (<code>dataframe</code>) زیر آماده کنید.
</font>
</p>

<div dir=rtl style="direction: rtl;text-align: center;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazir" size=3>
    
|ستون|توضیحات|
|------|---|
|Pool|وجود یا عدم وجود استخر|
|id|شناسه یکتای اقامتگاه|
    
</font>
</div>



<p dir="rtl" style="direction: rtl; text-align: justify; line-height: 200%; font-family: Vazir; font-size: medium;">
    نام دیتافریم باید <i>submission</i> باشد؛ در غیر این صورت، سامانه داوری نمی‌تواند تلاش شما را ارزیابی کند.
    <br>
    این دیتافریم شامل دو ستون با نام‌های <i>id</i> و <i>Pool</i> است و دارای ۸۶۲ سطر می‌باشد.
    <br>
    مقادیر موجود در ستون <i>Pool</i> شامل <i>True</i> یا <i>False</i> هستند.
    <br>
    برای هر سطر موجود در دیتافریم <i>test</i> باید یک مقدار پیش‌بینی‌شده داشته باشید.
    <br>
    جدول زیر، ۵ سطر ابتدایی دیتافریم <code>submission</code> را نشان می‌دهد. البته در جواب شما، مقادیر ستون <i>Pool</i> ممکن است متفاوت باشد.
</p>

<div style="text-align: center; line-height: 200%; font-family: Vazir; font-size: medium;">
    <font face="vazir" size="3">
        <table border="1" style="margin: 0 auto;">
            <tr>
                <th>id</th>
                <th>Pool</th>
            </tr>
            <tr>
                <td>3160664</td>
                <td>True</td>
            </tr>
            <tr>
                <td>3195184</td>
                <td>False</td>
            </tr>
            <tr>
                <td>3224078</td>
                <td>False</td>
            </tr>
            <tr>
                <td>3233712</td>
                <td>True</td>
            </tr>
            <tr>
                <td>3201449</td>
                <td>True</td>
            </tr>
        </table>
    </font>
</div>

In [14]:
test_datagen = ImageDataGenerator(rescale=1./255)
test_generator = test_datagen.flow_from_dataframe(
    dataframe=test_img_df,
    x_col='image_path',
    y_col=None,
    target_size=(128, 128),
    batch_size=32,
    class_mode=None,
    shuffle=False
)

test_preds = model.predict(test_generator)
test_img_df['pred_prob'] = test_preds

agg_preds = test_img_df.groupby('id')['pred_prob'].mean().reset_index()

agg_preds['pool'] = agg_preds['pred_prob'] > 0.5

submission = pd.DataFrame({'id': df_test['id']})
submission = submission.merge(agg_preds[['id', 'pool']], on='id', how='left')

submission['pool'] = submission['pool'].fillna(False)

submission['pool'] = submission['pool'].astype(bool)

print("Columns in submission:", submission.columns.tolist())
print(submission.head())

Found 8505 validated image filenames.
266/266 ━━━━━━━━━━━━━━━━━━━━ 49s 182ms/step
Columns in submission: ['id', 'pool']
        id   pool
0  3160664   True
1  3195184  False
2  3224078   True
3  3233712   True
4  3201449   True


<h2 dir=rtl align=right style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
<b>سلول جواب‌ساز</b>
</font>
</h2>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size=3>
    برای ساخته‌شدن فایل <code>result.zip</code> سلول زیر را اجرا کنید. توجه داشته باشید که پیش از اجرای سلول زیر تغییرات اعمال شده در نت‌بوک را ذخیره کرده باشید (<code>ctrl+s</code>) در غیر این صورت، در پایان مسابقه نمره شما به صفر تغییر خواهد کرد.
    <br>
    همچنین اگر از کولب برای اجرای این فایل نوت‌بوک استفاده می‌کنید، قبل از ارسال فایل <code>result.zip</code>، آخرین نسخه‌ی نوت‌بوک خود را دانلود کرده و داخل فایل ارسالی قرار دهید.
</font>

In [15]:
import zipfile
import joblib

if not os.path.exists(os.path.join(os.getcwd(), 'EstakhtemoonNasheh.ipynb')):
    %notebook -e EstakhtemoonNasheh.ipynb

def compress(file_names):
    print("File Paths:")
    print(file_names)
    compression = zipfile.ZIP_DEFLATED
    with zipfile.ZipFile("result.zip", mode="w") as zf:
        for file_name in file_names:
            zf.write('./' + file_name, file_name, compress_type=compression)
            
submission.to_csv('submission.csv', index=False)
file_names = ['EstakhtemoonNasheh.ipynb', 'submission.csv']
compress(file_names)

File Paths:
['EstakhtemoonNasheh.ipynb', 'submission.csv']
